# Semana 45 · ¿Por qué se curva la luz cerca de una masa?

Este cuaderno acompaña a la sesión de proyecto de la semana 45: desviaciones y radios de Einstein, el trazado de rayos de una lente gravitacional, la curva de luz de una microlente y la precesión de Mercurio.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Desviaciones y radios de Einstein

$$\alpha = \frac{4GM}{c^2 b}, \qquad \theta_E = \sqrt{\frac{4GM}{c^2}\,\frac{D_{ls}}{D_l D_s}}.$$

Para los radios de Einstein usamos, por sencillez, distancias euclídeas con $D_{ls} = D_s - D_l$ (en cosmología hacen falta otras definiciones de distancia, pero el orden de magnitud no cambia).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G, c = 6.674e-11, 2.998e8
M_SOL = 1.989e30
PC = 3.0857e16
RAD_A_ARCSEC = 206264.8

def desviacion(M, b):
    return 4 * G * M / (c**2 * b)

def radio_einstein(M, D_l, D_s):
    return np.sqrt(4 * G * M / c**2 * (D_s - D_l) / (D_l * D_s))

print(f"rozando el Sol: {desviacion(M_SOL, 6.957e8) * RAD_A_ARCSEC:.3f}″")
print(f"rozando Júpiter: {desviacion(1.898e27, 7.149e7) * RAD_A_ARCSEC:.4f}″")
print(f"rozando una enana blanca (0,6 M☉, 8700 km): {desviacion(0.6 * M_SOL, 8.7e6) * RAD_A_ARCSEC:.1f}″")
print(f"rozando una estrella de neutrones (1,4 M☉, 12 km): {np.degrees(desviacion(1.4 * M_SOL, 1.2e4)):.0f}° (¡la fórmula ya no vale!)")
print()
casos = [("estrella (0,3 M☉) a 4 kpc, fuente a 8 kpc", 0.3 * M_SOL, 4e3 * PC, 8e3 * PC),
         ("galaxia (10¹² M☉) a 1 Gpc, fuente a 2 Gpc", 1e12 * M_SOL, 1e9 * PC, 2e9 * PC),
         ("cúmulo (10¹⁵ M☉) a 1 Gpc, fuente a 3 Gpc", 1e15 * M_SOL, 1e9 * PC, 3e9 * PC)]
for nombre, M, Dl, Ds in casos:
    tE = radio_einstein(M, Dl, Ds) * RAD_A_ARCSEC
    texto = f"{tE * 1000:.2f} milisegundos de arco" if tE < 0.1 else f"{tE:.1f}″"
    print(f"{nombre:<46} θ_E = {texto}")

Para una estrella de neutrones, la desviación calculada sale de decenas de grados: la fórmula, que vale para desviaciones pequeñas, ya no sirve, y hay que usar la relatividad general completa. Los radios de Einstein van de milisegundos de arco (estrellas: imposible separar las imágenes, solo se nota el aumento de brillo) a decenas de segundos de arco (cúmulos: arcos espectaculares).

## 2. Trazar rayos a través de una lente

El método se llama **trazado inverso de rayos**: para cada píxel de la imagen que veríamos (ángulo $\vec\theta$), calculamos de qué punto de la fuente viene la luz (ángulo $\vec\beta$) con la **ecuación de la lente** y miramos cuánto brilla la fuente allí. Para una lente puntual,

$$\vec\beta = \vec\theta - \theta_E^2\,\frac{\vec\theta}{|\vec\theta|^2}.$$

Para una galaxia, un modelo sencillo y bastante realista es la **esfera isoterma singular** (la de las curvas de rotación planas de la semana 43), con

$$\vec\beta = \vec\theta - \theta_E\,\frac{\vec\theta}{|\vec\theta|}.$$

La galaxia de fondo es una mancha gaussiana un poco alargada. Trabajamos en unidades de $\theta_E$.

In [ ]:
n = 600
eje = np.linspace(-2.5, 2.5, n)
TX, TY = np.meshgrid(eje, eje)
radio = np.hypot(TX, TY) + 1e-12

def fuente(bx, by, x0, y0, s=0.12):
    return np.exp(-0.5 * (((bx - x0) / (1.5 * s)) ** 2 + ((by - y0) / s) ** 2))

def lente_puntual(tx, ty):
    return tx - tx / radio**2, ty - ty / radio**2

def lente_isoterma(tx, ty):
    return tx - tx / radio, ty - ty / radio

fig, axs = plt.subplots(2, 3, figsize=(14, 9))
for fila, (nombre, lente) in enumerate([("lente puntual", lente_puntual), ("galaxia (esfera isoterma)", lente_isoterma)]):
    BX, BY = lente(TX, TY)
    for col, x0 in enumerate([0.0, 0.15, 0.6]):
        ax = axs[fila, col]
        ax.imshow(fuente(BX, BY, x0, 0.0), extent=(-2.5, 2.5, -2.5, 2.5), origin="lower", cmap="magma")
        ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="cyan", ls=":", lw=0.8))
        ax.plot(x0, 0, "w+", ms=10)
        ax.set_title(f"{nombre}\nfuente desplazada {x0} θ_E", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Imágenes de una galaxia de fondo (+ = posición real de la fuente; círculo = radio de Einstein)")
plt.tight_layout(); plt.show()

Con la fuente justo detrás de la lente sale un **anillo de Einstein**. Si se desplaza un poco, el anillo se rompe en dos **arcos**; si se desplaza más, quedan **dos imágenes**, una brillante fuera del círculo y otra débil dentro, al otro lado. Fíjate en que las imágenes son más grandes que la fuente: la lente también **amplifica**. Por eso los cúmulos de galaxias sirven como telescopios naturales para ver galaxias lejanísimas.

## 3. La curva de luz de una microlente

Cuando una estrella pasa por delante de otra, las dos imágenes están tan juntas (milisegundos de arco) que no se separan, pero su brillo total aumenta en un factor

$$A(u) = \frac{u^2 + 2}{u\sqrt{u^2 + 4}},$$

donde $u$ es la separación entre la lente y la fuente en unidades de $\theta_E$, que cambia con el tiempo al moverse las estrellas: $u(t) = \sqrt{u_0^2 + ((t - t_0)/t_E)^2}$. Es la **curva de Paczyński**, por el astrónomo polaco Bohdan Paczyński, que propuso en 1986 buscar así objetos oscuros de la galaxia. Añadimos un planeta como una pequeña segunda lente que pasa cerca del camino de la fuente (modelo muy simplificado: sumamos su efecto como si fuera independiente).

In [ ]:
def amplificacion(u):
    return (u**2 + 2) / (u * np.sqrt(u**2 + 4))

t = np.linspace(-40, 40, 4000)                 # días
t0, tE, u0 = 0.0, 15.0, 0.2
u_estrella = np.sqrt(u0**2 + ((t - t0) / tE) ** 2)
A_estrella = amplificacion(u_estrella)

# un planeta con una masa mil veces menor: su radio de Einstein es √1000 ≈ 32 veces menor,
# y su paso dura unas 32 veces menos; sumamos su efecto aparte (aproximación muy simplificada)
q, t_p = 1e-3, 12.0
tE_p = tE * np.sqrt(q)
u_planeta = np.sqrt(1.0**2 + ((t - t_p) / tE_p) ** 2)
A_total = A_estrella + (amplificacion(u_planeta) - 1)

plt.figure(figsize=(10, 4.5))
plt.plot(t, A_estrella, label="solo la estrella")
plt.plot(t, A_total, alpha=0.8, label="estrella con planeta (esquema)")
plt.xlabel("días desde el máximo"); plt.ylabel("amplificación"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"amplificación máxima: {A_estrella.max():.1f} veces, es decir, {2.5 * np.log10(A_estrella.max()):.1f} magnitudes más brillante")

La curva de la estrella es simétrica y suave, y dura semanas. El planeta añade un pico breve, de horas o un día. Los cartografiados de microlentes, que vigilan cada noche cientos de millones de estrellas del centro de la galaxia, han encontrado así más de doscientos planetas.

## 4. Para ir más lejos: la precesión de Mercurio

$$\Delta\varphi = \frac{6\pi GM}{c^2 a (1 - e^2)}\ \text{por vuelta}.$$

In [ ]:
GM_sol = 1.32712e20
planetas = [("Mercurio", 5.791e10, 0.2056, 87.969), ("Venus", 1.0821e11, 0.0068, 224.70),
            ("Tierra", 1.496e11, 0.0167, 365.256), ("Marte", 2.2794e11, 0.0934, 686.98)]
for nombre, a, e, periodo in planetas:
    por_vuelta = 6 * np.pi * GM_sol / (c**2 * a * (1 - e**2))
    por_siglo = por_vuelta * 36525 / periodo * RAD_A_ARCSEC
    print(f"{nombre:<9} {por_siglo:6.2f}″ por siglo")

## 5. Experimenta

1. En el trazado de rayos, cambia la forma de la fuente (más alargada, o dos manchas) y su posición en vertical. ¿Cuántas imágenes aparecen con la lente isoterma?
2. Calcula la amplificación total de las dos imágenes con $A(u)$ para $u = 1$, $0{,}1$ y $0{,}01$. ¿Qué pasa cuando la alineación es casi perfecta?
3. ¿Qué radio de Einstein tendría el Sol para una estrella lejana vista desde la Tierra? (Pista: con $D_l = 1$ ua, $\theta_E$ sale menor que el radio del propio Sol, y la luz pasaría por dentro. ¿A qué distancia del Sol habría que ponerse para que el Sol funcionara como lente? Es la idea de las misiones que proponen usar el Sol como telescopio, a unas 550 ua.)

In [ ]:
# Tu experimento aquí